# 06 · Spielunterbrechungen und Spielfortsetzungen

**Frage:** Können wir die in Notebook 05 beobachteten D-Fehler durch eine zusätzliche Zustandslogik reduzieren, ohne laufendes Spiel stark zu zerschneiden?

**Hypothese:** Ein Ball kann nach einem Pfiff weiterrollen oder vor einem Abstoß geholt werden. Kontakt und Bewegung alleine belegen kein Live-Spiel. Wir untersuchen Audio, Feldgrenzen und das Bewegungsverhalten mehrerer Spieler gemeinsam; den tatsächlichen Neustart behandeln wir separat.

Dies ist ein erster explorativer Versuch mit festgelegten Schwellen, kein trainierter Pfiffdetektor und keine vollständige Reproduktion des Papers. Das Paper setzt eine Live-/Dead-Information bereits voraus. Die H/A-Kontaktregel aus Notebook 04 wird exakt reproduziert und mit einer erweiterten D-Maske erneut abgespielt.

In [ ]:
# Standardbibliothek
from pathlib import Path
from io import BytesIO
from zipfile import ZipFile
from tempfile import TemporaryDirectory
from importlib.metadata import version
import hashlib
import json
import shutil
import subprocess
import sys
import xml.etree.ElementTree as ET

# Daten, Video und Darstellung
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from IPython.display import display, Markdown, Video

## 1. Versuchsprotokoll

- Die beiden bekannten D-Fehler und zwei Live-Gegenbeispiele aus Notebook 05 sind **Entwicklungsdiagnostik**. Die öffentliche GT ist deshalb kein unberührter Test mehr.
- Drei weitere Kontrollclips werden allein aus den automatischen Tags außerhalb der öffentlichen fünf Minuten gewählt: jeweils der erste Freistoß, Abstoß und Einwurf nach diesem Ausschnitt. Ihre tatsächlichen D-Grenzen sind noch nicht manuell verifiziert. Ein Tagname ist keine Referenzannotation.
- Es gibt **eine** neue Regelkonfiguration, keine Schwellenwertsuche. Die Parameter sind runde, überprüfbare Arbeitshypothesen nach Sichtung der bekannten Fehler. Vorhersagen werden vor dem erneuten Einlesen der GT erzeugt und gespeichert.
- Vergleich: `baseline_smooth50`, gespeicherte Kontaktregel aus Notebook 04 und Kontaktregel mit zusätzlichem D-Zustand. Offizieller Validator und Evaluator bleiben unverändert.
- Erfolg verlangt nicht nur mehr D-Recall: Wir zeigen auch D-Precision, neu unterbrochene Live-Abschnitte und die Rangmetrik `spellF1`. Auch ein negatives Ergebnis bleibt erhalten.

Die aus Notebook 03 bekannte Strafraumaktion wird ergänzend als Live-Gegenprobe aufgenommen. Diese Ergänzung folgt auf den ersten Scorelauf; sie ist keine unabhängige Validierung und verändert keine Regelparameter.

In [ ]:
PROJECT_DIR = next(p for p in (Path.cwd(), *Path.cwd().parents)
                   if (p / "tactical/BAR-ATH_tactical_atd.xml").is_file())
META_PATH = PROJECT_DIR / "tactical/BAR-ATH_tactical_atd.xml"
RAW_PATH = PROJECT_DIR / "tactical/BAR-ATH_tactical_atd.txt"
VIDEO_PATH = PROJECT_DIR / "tactical/BAR-ATH_tactical.mp4"
TAGS_PATH = PROJECT_DIR / "tactical/BAR-ATH_tactical_smart_tagging.xml"
ARCHIVE = PROJECT_DIR / "reference/mtah2027_official_3de6b60.zip"
PREVIOUS_DIR = PROJECT_DIR / "artifacts/upstream/contact_classification"
PREVIOUS_MANIFEST = PREVIOUS_DIR / "run_manifest.json"
BASELINE_PATH = PREVIOUS_DIR / "submissions/baseline_smooth50.csv"
CONTACT_PATH = PREVIOUS_DIR / "submissions/contact_state_shared_D.csv"
OUTPUT_DIR = PROJECT_DIR / "output/stoppage_detection"
SUBMISSION_DIR, EVALUATION_DIR = OUTPUT_DIR / "submissions", OUTPUT_DIR / "evaluation"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)
config = json.loads(PREVIOUS_MANIFEST.read_text())
INPUT_HASHES = {str(p.relative_to(PROJECT_DIR)): hashlib.sha256(p.read_bytes()).hexdigest()
                for p in [PREVIOUS_MANIFEST, BASELINE_PATH, CONTACT_PATH, ARCHIVE]}
assert INPUT_HASHES[str(ARCHIVE.relative_to(PROJECT_DIR))] == config["archive_sha256"]
FPS, N_FRAMES, PITCH = config["fps"], config["n_frames"], np.array(config["pitch_m"])
HALVES = [("1. Halbzeit", 11185, 90651), ("2. Halbzeit", 110808, 190197)]
MEASUREMENT_SIGMA_M, PROCESS_SIGMA_M_S2 = config["measurement_sigma_m"], config["process_sigma_m_s2"]
NEAR_DISTANCE_M, PEAK_ACCELERATION_M_S2 = config["near_distance_m"], config["peak_acceleration_m_s2"]
MIN_RUN_SECONDS, EDGE_GUARD_SECONDS = config["min_run_seconds"], config["edge_guard_seconds"]
BASELINE_NAME, CONTACT_NAME, METHOD_NAME = "baseline_smooth50", "contact_state_shared_D", "contact_state_stoppage_v1"
FFMPEG, FFPROBE = shutil.which("ffmpeg"), shutil.which("ffprobe")
if FFMPEG is None or FFPROBE is None:
    raise RuntimeError("Dieses Notebook benötigt die bereits lokal verwendeten Programme ffmpeg und ffprobe.")
D_PARAMS = {
    "audio_sample_rate": 8000, "audio_band_hz": [1800, 3800], "tone_peak_hz": [2000, 3800],
    "tone_ratio": 0.20, "highband_ratio": 0.30, "min_audio_dbfs": -80.0,
    "tone_frequency_change_hz": 200.0, "seed_min_seconds": 0.12,
    "outside_margin_m": 0.5, "player_difference_seconds": 0.4,
    "max_valid_player_speed_m_s": 15.0, "min_players": 12, "active_player_speed_m_s": 2.0,
    "quiet_active_fraction": 0.25, "quiet_seconds": 1.0, "slow_ball_m_s": 3.0,
    "confirmation_seconds": 12.0, "ready_ball_m_s": 1.0, "ready_seconds": 0.4,
    "ready_history_seconds": 5.0, "launch_ball_m_s": 4.0, "launch_seconds": 0.2,
    "restart_lookahead_seconds": 3.0, "restart_active_fraction": 0.20,
    "restart_active_seconds": 0.4, "max_added_dead_seconds": 45.0,
}
LABEL_ORDER, COLORS = ["H", "A", "D", "X"], ["#1764ab", "#c83e41", "#999999", "#eee5ce"]
plt.rcParams.update({"figure.dpi": 105, "axes.spines.top": False, "axes.spines.right": False})
display(pd.Series(D_PARAMS, name="Vor GT-Vergleich festgelegte D-Parameter").to_frame())
print(f"Python {sys.version.split()[0]} | NumPy {version('numpy')} | pandas {version('pandas')}")

## 2. Vorhersagen und Tracking laden

Wir verwenden ausschließlich die vorhandenen Tactical-Daten. Rohframe minus eins ergibt den Videoindex; es wird kein zusätzlicher Tracking-Offset angewandt. Die 46 Track-Slots sind keine dauerhaften Spieleridentitäten. Geschwindigkeiten werden aus metrischen Positionen berechnet, nicht aus dem uneindeutigen Roh-Geschwindigkeitskanal.

In [ ]:
with ZipFile(ARCHIVE) as official:
    ARCHIVE_ROOT = official.namelist()[0]
    baseline_table = pd.read_csv(BytesIO(official.read(ARCHIVE_ROOT + f"evaluation/results/{BASELINE_NAME}.csv")))
assert np.array_equal(baseline_table["frame"], np.arange(N_FRAMES)), "Baseline und Videoindex sind nicht framegleich."
baseline_labels = baseline_table["label"].to_numpy()
saved_baseline = pd.read_csv(BASELINE_PATH)
saved_contact = pd.read_csv(CONTACT_PATH)
assert saved_baseline.equals(baseline_table), "Gespeicherte und offizielle Baseline unterscheiden sich."
assert np.array_equal(saved_contact["frame"], np.arange(N_FRAMES)), "Kontaktvorhersagen sind nicht framegleich."
contact_labels = saved_contact["label"].to_numpy()

root = ET.parse(META_PATH).getroot()
team_nodes = root.findall("./Metadata/Teams/Team")
team_map = {t.get("id"): ("H" if i == 0 else "A", t.findtext("Name")) for i, t in enumerate(team_nodes)}
channel_to_track = {c.get("id"): c.get("playerId") for c in root.findall("./Metadata/PlayerChannels/PlayerChannel")}
spec = root.find("./DataFormatSpecifications/DataFormatSpecification")
track_ids = [channel_to_track[g.find("PlayerChannelRef").get("playerChannelId")]
             for g in spec.findall("./SplitRegister/SplitRegister")]
tracks = pd.DataFrame([{"track_id": p.get("id"), "side": team_map[p.get("teamId")][0],
                        "team": team_map[p.get("teamId")][1]} for p in root.findall("./Metadata/Players/Player")]).set_index("track_id").loc[track_ids]
positions = np.empty((N_FRAMES, len(track_ids) + 1, 2), dtype=float)
frame_ids = np.empty(N_FRAMES, dtype=int)
with RAW_PATH.open() as handle:
    for row, line in enumerate(handle):
        raw_frame, players_raw, ball_raw = line.strip().split(":")
        frame_ids[row] = int(raw_frame) - 1
        positions[row] = np.fromstring(players_raw.replace(";", ",") + "," + ball_raw, sep=",").reshape(-1, 3)[:, :2] * PITCH
assert row + 1 == N_FRAMES and np.array_equal(frame_ids, np.arange(N_FRAMES)), "Tracking-Zeitachse stimmt nicht mit der Baseline überein."
player_xy, ball_xy = positions[:, :-1], positions[:, -1]
ball_seen = np.isfinite(ball_xy).all(axis=1)
print(f"{N_FRAMES:,} Frames, {len(tracks)} Track-Slots; Ball in {100 * ball_seen.mean():.1f} % des Gesamtvideos beobachtet.")
display(tracks.groupby(["side", "team"]).size().rename("Track-Slots").to_frame())

## 3. Audiospur überprüfen und tonale Hinweise ableiten

`ffprobe` zeigt, ob eine Audiospur existiert und ob ihre Startzeit zur Videozeit passt. Audio wird mono mit 8 kHz dekodiert. Je 40-ms-Videoframe berechnen wir ein Hann-Fensterspektrum: Energieanteil 1,8–3,8 kHz, Energie in drei Frequenzbins um den stärksten Ton sowie dessen Frequenz. Die drei Bins umfassen 75 Hz.

Ein **tonaler Kandidat** verlangt mindestens 0,12 s Unterstützung, einen stabilen Peak bei 2–3,8 kHz und die oben festgelegten Energieanteile. Das erkennt einen schmalen Ton, **nicht dessen Quelle**: Publikumspfiffe und andere Geräusche können dieselbe Bedingung erfüllen. Lautstärke ist dBFS der Datei und kein akustischer Schalldruckpegel. Wir setzen Audio nie mit einem Schiedsrichterpfiff gleich.

Audio und Video beginnen in dieser Datei bei derselben Containerzeit; eine mögliche inhaltliche Ton-/Bild-Verzögerung ist damit noch nicht gemessen. Die letzten unvollständigen Audioframes bleiben fehlend.

In [ ]:
probe = json.loads(subprocess.run([FFPROBE, "-v", "error", "-show_entries",
    "stream=index,codec_type,codec_name,start_time,sample_rate,channels,duration", "-of", "json", str(VIDEO_PATH)],
    capture_output=True, text=True, check=True).stdout)
streams = pd.DataFrame(probe["streams"])
display(streams.fillna(""))
audio_streams = streams.loc[streams["codec_type"].eq("audio")]
audio_columns = ["audio_dbfs", "highband_ratio", "tone_ratio", "tone_hz"]
audio_features = np.full((N_FRAMES, len(audio_columns)), np.nan)
if not audio_streams.empty:
    video_start = float(streams.loc[streams["codec_type"].eq("video"), "start_time"].iloc[0])
    audio_start = float(audio_streams["start_time"].iloc[0])
    if abs(audio_start - video_start) > 1 / FPS:
        raise ValueError("Audio und Video beginnen nicht framegleich; vor Auswertung explizit ausrichten.")
    sample_rate = D_PARAMS["audio_sample_rate"]
    width = sample_rate // FPS
    frequencies = np.fft.rfftfreq(width, 1 / sample_rate)
    spectrum_mask = (frequencies >= 200) & (frequencies <= 3900)
    band_mask = (frequencies >= D_PARAMS["audio_band_hz"][0]) & (frequencies <= D_PARAMS["audio_band_hz"][1])
    with TemporaryDirectory(prefix="mtah06-audio-") as temporary:
        decoded = Path(temporary) / "audio.f32"
        subprocess.run([FFMPEG, "-hide_banner", "-loglevel", "error", "-i", str(VIDEO_PATH),
                        "-map", "0:a:0", "-ac", "1", "-ar", str(sample_rate), "-f", "f32le", str(decoded)],
                       capture_output=True, text=True, check=True)
        samples = np.memmap(decoded, dtype="<f4", mode="r")
        n_audio = min(N_FRAMES, len(samples) // width)
        for first in range(0, n_audio, 1500):
            last = min(first + 1500, n_audio)
            windows = np.array(samples[first * width:last * width]).reshape(-1, width)
            power = np.abs(np.fft.rfft(windows * np.hanning(width), axis=1)) ** 2
            total_power = power[:, spectrum_mask].sum(axis=1) + 1e-20
            band_power = power[:, band_mask]
            peak = band_power.argmax(axis=1)
            tone_power = np.array([row[max(0, j - 1):j + 2].sum() for row, j in zip(band_power, peak)])
            audio_features[first:last] = np.c_[
                20 * np.log10(np.sqrt(np.mean(windows ** 2, axis=1)) + 1e-12),
                band_power.sum(axis=1) / total_power, tone_power / total_power, frequencies[band_mask][peak]]
        del samples
audio = pd.DataFrame(audio_features, index=pd.Index(frame_ids, name="frame"), columns=audio_columns)
print(f"Audio-Merkmale für {audio['audio_dbfs'].notna().sum():,} von {N_FRAMES:,} Videoframes.")
display(audio[["audio_dbfs", "highband_ratio", "tone_ratio"]].quantile([0.1, 0.5, 0.9, 0.99]).round(3))

## 4. Bisherige Kontakte exakt rekonstruieren

Der RTS-Glätter und die Kontaktkandidaten sind unverändert aus Notebook 04 übernommen. Ergänzt wird nur die Geschwindigkeit des geglätteten Balls als Diagnosesignal. Balllücken werden nicht interpoliert; Ableitungen werden nur in zusammenhängenden beobachteten Läufen berechnet. Der RTS-Glätter nutzt Zukunftsinformation, wie bereits in Notebook 04.

In [ ]:
def smooth_ball_rts(points, fps, measurement_sigma, process_sigma):
    dt = 1 / fps
    transition = np.eye(4)
    transition[0, 2] = transition[1, 3] = dt
    observation = np.eye(2, 4)
    acceleration_map = np.r_[np.eye(2) * dt**2 / 2, np.eye(2) * dt]
    process_noise = acceleration_map @ acceleration_map.T * process_sigma**2
    measurement_noise = np.eye(2) * measurement_sigma**2
    state = np.r_[points[0], (points[1] - points[0]) / dt]
    covariance = np.diag([measurement_sigma**2, measurement_sigma**2, 100.0, 100.0])
    predicted = np.empty((len(points), 4))
    filtered = np.empty_like(predicted)
    predicted_cov = np.empty((len(points), 4, 4))
    filtered_cov = np.empty_like(predicted_cov)

    for i, point in enumerate(points):
        if i:
            state = transition @ state
            covariance = transition @ covariance @ transition.T + process_noise
        predicted[i], predicted_cov[i] = state, covariance
        gain = np.linalg.solve(observation @ covariance @ observation.T + measurement_noise,
                               (covariance @ observation.T).T).T
        state = state + gain @ (point - observation @ state)
        residual_map = np.eye(4) - gain @ observation
        covariance = residual_map @ covariance @ residual_map.T + gain @ measurement_noise @ gain.T
        filtered[i], filtered_cov[i] = state, covariance

    smoothed = filtered.copy()
    for i in range(len(points) - 2, -1, -1):
        gain = np.linalg.solve(predicted_cov[i + 1].T, (filtered_cov[i] @ transition.T).T).T
        smoothed[i] += gain @ (smoothed[i + 1] - predicted[i + 1])
    return smoothed

In [ ]:
starts = np.flatnonzero(ball_seen & ~np.r_[False, ball_seen[:-1]])
ends = np.flatnonzero(ball_seen & ~np.r_[ball_seen[1:], False])
signals = pd.DataFrame(np.nan, index=pd.Index(frame_ids, name="frame"),
                       columns=["acceleration_m_s2", "nearest_distance_m", "nearest_offset", "speed_m_s"])
signals["eligible"] = False
edge_guard = round(EDGE_GUARD_SECONDS * FPS)
for first, last in zip(starts, ends):
    idx = np.arange(first, last + 1)
    if len(idx) / FPS < MIN_RUN_SECONDS:
        continue
    state = smooth_ball_rts(ball_xy[idx], FPS, MEASUREMENT_SIGMA_M, PROCESS_SIGMA_M_S2)
    acceleration = np.linalg.norm(np.gradient(state[:, 2:], 1 / FPS, axis=0), axis=1)
    distances = np.linalg.norm(player_xy[idx] - state[:, None, :2], axis=2)
    distances = np.where(np.isfinite(distances), distances, np.inf)
    nearest_distance = distances.min(axis=1)
    nearest_offset = np.where(np.isfinite(nearest_distance), distances.argmin(axis=1), np.nan)
    nearest_distance[~np.isfinite(nearest_distance)] = np.nan
    signals.loc[idx, ["acceleration_m_s2", "nearest_distance_m", "nearest_offset", "speed_m_s"]] = np.c_[acceleration, nearest_distance, nearest_offset, np.linalg.norm(state[:, 2:], axis=1)]
    if len(idx) > 2 * edge_guard:
        signals.loc[idx[edge_guard:-edge_guard], "eligible"] = True

assert signals.loc[~ball_seen, "acceleration_m_s2"].isna().all(), "Beschleunigungen dürfen Balllücken nicht überbrücken."
local_peak = signals["acceleration_m_s2"].gt(signals["acceleration_m_s2"].shift()) & signals["acceleration_m_s2"].ge(signals["acceleration_m_s2"].shift(-1))
signals["candidate"] = (signals["eligible"] & local_peak & signals["acceleration_m_s2"].ge(PEAK_ACCELERATION_M_S2)
                        & signals["nearest_distance_m"].le(NEAR_DISTANCE_M))
candidate_side = np.full(N_FRAMES, "", dtype="<U1")
candidate_mask = signals["candidate"].to_numpy()
candidate_side[candidate_mask] = tracks["side"].to_numpy()[signals.loc[candidate_mask, "nearest_offset"].to_numpy(dtype=int)]
print(f"{signals['eligible'].sum():,} Frames für die Peakdiagnose auswertbar; {candidate_mask.sum():,} Kandidaten im Gesamtvideo.")

In [ ]:
def classify_from_contacts(baseline, event_side, halves):
    prediction = baseline.copy()
    uses_contact_state = np.zeros(len(baseline), dtype=bool)
    for _, first, last in halves:
        last_team = None
        for frame in range(first, last + 1):
            if baseline[frame] == "D":
                last_team = None
                continue
            if event_side[frame] in ("H", "A"):
                last_team = event_side[frame]
            if last_team is not None:
                prediction[frame] = last_team
                uses_contact_state[frame] = True
    return prediction, uses_contact_state

replayed_contact, _ = classify_from_contacts(baseline_labels, candidate_side, HALVES)
assert np.array_equal(replayed_contact, contact_labels), "Die bisherige H/A-Methode wird nicht exakt reproduziert."
print(f"Alle {N_FRAMES:,} Kontaktvorhersagen aus Notebook 04 exakt reproduziert.")

## 5. Gemeinsame Bewegung und Feldgrenzen

Spielergeschwindigkeit: Positionsdifferenz über 0,4 s innerhalb desselben Track-Slots. Fehlende Endpunkte und Geschwindigkeiten über 15 m/s werden als nicht auswertbar behandelt. Auswertbare Spieler sind keine Zufallsstichprobe; die sichtbare Gruppe hängt von Kamera und Tracking ab. Für Gruppenhinweise verlangen wir mindestens zwölf auswertbare Slots. „Aktiv“ bedeutet hier lediglich schneller als 2 m/s.

„Außerhalb“ ist eine beobachtete Ballposition mehr als 0,5 m jenseits des Rechtecks. Bei einem hohen Ball ist die 2D-Position kein sicherer Beleg für das Überschreiten der Linie. Wir verlangen deshalb anschließend eine gemeinsame Beruhigung. Fehlender Ball alleine löst keine neue Unterbrechung aus.

Eine beobachtete Ballposition ohne auswertbare Geschwindigkeit (etwa ein sehr kurzer Tracklauf) wird nicht als fehlender Ball behandelt.

In [ ]:
offset = round(D_PARAMS["player_difference_seconds"] * FPS / 2)
player_speed = np.full(player_xy.shape[:2], np.nan)
player_speed[offset:-offset] = np.linalg.norm(player_xy[2 * offset:] - player_xy[:-2 * offset], axis=2) * FPS / (2 * offset)
player_speed[player_speed > D_PARAMS["max_valid_player_speed_m_s"]] = np.nan
visible = np.isfinite(player_speed).sum(axis=1)
signals["visible_players"] = visible
signals["active_fraction"] = np.where(visible >= D_PARAMS["min_players"],
    (player_speed > D_PARAMS["active_player_speed_m_s"]).sum(axis=1) / np.maximum(visible, 1), np.nan)
signals["median_player_speed_m_s"] = pd.DataFrame(player_speed).median(axis=1).to_numpy()
margin = D_PARAMS["outside_margin_m"]
signals["outside"] = ball_seen & ((ball_xy[:, 0] < -margin) | (ball_xy[:, 0] > PITCH[0] + margin)
                                 | (ball_xy[:, 1] < -margin) | (ball_xy[:, 1] > PITCH[1] + margin))
signals = signals.join(audio)
tag_root = ET.parse(TAGS_PATH).getroot()
set_pieces = pd.DataFrame([
    {"id": int(e.findtext("ID")), "start_s": float(e.findtext("start")), "end_s": float(e.findtext("end")),
     **{label.findtext("group"): label.findtext("text") for label in e.findall("label")}}
    for e in tag_root.findall("./ALL_INSTANCES/instance") if e.findtext("code") == "SET PIECES"
])
display(pd.Series({"Ball beobachtet (%)": 100 * ball_seen.mean(),
                   "Gruppenbewegung auswertbar (%)": 100 * signals["active_fraction"].notna().mean(),
                   "Ballposition außerhalb (%)": 100 * signals["outside"].mean(),
                   "Automatische Standardintervalle": len(set_pieces)}).round(2).to_frame("Gesamtvideo"))

## 6. Entwicklungs- und Kontrollszenen vor dem Scorevergleich festlegen

Die ersten beiden Szenen sind die bekannten langen D-Fehler; die nächsten beiden und der ergänzte Strafraumclip zeigen laut bisheriger Referenz laufendes Spiel. Die drei zusätzlichen Clips werden ohne Betrachtung der neuen Vorhersagen und ohne öffentliche GT ausgewählt. Die Standard-Tags dienen ausschließlich zur Auswahl und Anzeige, **nicht** als D-Maske oder Feature der Regel.

Für die Kontrollclips wird eine separate Sichtungsdatei vorbereitet. Leere Referenzfelder bleiben leer: Es werden keine tatsächlichen Pfiff- oder Wiederanpfiffzeitpunkte erfunden. Die neuen Vorhersageintervalle stehen später getrennt davon in `state_intervals.csv`.

In [ ]:
scene_rows = [
    {"scene_id": "dev_foul", "role": "Entwicklung", "reason": "Bekannte Unterbrechung nach Aktion", "first": 15750, "last": 16650},
    {"scene_id": "dev_goal_kick", "role": "Entwicklung", "reason": "Bekannte Ballrückholung / Fortsetzung", "first": 17875, "last": 18675},
    {"scene_id": "dev_pass", "role": "Live-Gegenbeispiel", "reason": "Bekannte Passszene", "first": 14050, "last": 14400},
    {"scene_id": "dev_duel", "role": "Live-Gegenbeispiel", "reason": "Bekannte schnelle Teamwechsel", "first": 14750, "last": 15000},
    {"scene_id": "dev_box_live", "role": "Ergänzende Entwicklungsdiagnostik", "reason": "Bekannte Strafraumaktion und Weiterspielen", "first": 11550, "last": 11900},
]
for kind, key in [("Free Kick", "free_kick"), ("Goal Kick", "goal_kick"), ("Throw-in", "throw_in")]:
    tag = set_pieces.loc[set_pieces["Type"].eq(kind) & set_pieces["start_s"].gt(18685 / FPS)].sort_values("start_s").iloc[0]
    scene_rows.append({"scene_id": "control_" + key, "role": "Neue Sichtung, ohne Referenz",
                       "reason": f"Automatischer Tag {int(tag['id'])}: {kind}",
                       "first": max(0, round((tag["start_s"] - 8) * FPS)),
                       "last": min(N_FRAMES - 1, round((tag["end_s"] + 4) * FPS) - 1)})
scenes = pd.DataFrame(scene_rows).set_index("scene_id")
scenes["start_s"], scenes["end_s"] = scenes["first"] / FPS, (scenes["last"] + 1) / FPS
scenes["duration_s"] = (scenes["last"] - scenes["first"] + 1) / FPS
display(scenes[["role", "reason", "start_s", "end_s", "duration_s"]].round(2))
annotation_path = OUTPUT_DIR / "control_review.csv"
if not annotation_path.exists():
    review = scenes.loc[scenes.index.str.startswith("control_")].copy()
    review["reviewed"] = False
    for column in ["dead_start_frame", "restart_frame", "additional_dead_intervals", "live_counterexample_frames", "uncertainty_frames", "notes"]:
        review[column] = pd.NA
    review.to_csv(annotation_path)
print(f"Sichtungsdatei: {annotation_path.relative_to(PROJECT_DIR)}; bestehende Einträge werden nicht überschrieben.")

## 7. Erste Zustandsregel: Stoppbeginn und Spielfortsetzung trennen

**Stoppbeginn:** Ein stabiler tonaler Kandidat oder eine mindestens 0,12 s beobachtete Ballposition außerhalb liefert einen Hinweis. Innerhalb von zwölf Sekunden muss die sichtbare Gruppe mindestens eine Sekunde lang ruhig werden (höchstens 25 % schneller als 2 m/s). Am Bestätigungsframe muss der Ball langsam sein (höchstens 3 m/s) oder fehlen. Die fehlende Ballposition ist nur zusammen mit dem vorherigen Hinweis und der Gruppenberuhigung zulässig. Die D-Phase wird dann rückwirkend beim Hinweis begonnen.

**Spielfortsetzung:** Der Ball muss in den letzten fünf Sekunden mindestens 0,4 s langsam gewesen sein (höchstens 1 m/s). Ein unveränderter Kontaktkandidat innerhalb des Felds muss von mindestens 0,2 s schneller Ballbewegung (mindestens 4 m/s) und innerhalb von drei Sekunden von mindestens 0,4 s erneuter Gruppenbewegung (mindestens 20 % aktiv) gefolgt werden. Ballholen und Platzieren alleine sollen dadurch nicht genügen. Auch diese Regeln können eine kurze Ausführung übersehen oder Ballrückholung mit einem echten Neustart verwechseln.

**Begrenzung:** Wenn in 45 s kein Neustart bestätigt wird, endet nur die zusätzliche D-Phase durch einen ausdrücklich protokollierten *Fallback* zur bisherigen Methode. Das ist kein erkannter Wiederanpfiff. Halbzeitgrenzen setzen den Zustand zurück. Bestehende Baseline-D-Frames bleiben erhalten. Die H/A-Regel wird nach jeder D-Phase zurückgesetzt und bis zum nächsten Kandidaten mit der Baseline initialisiert.

Die Regel ist **offline**: RTS-Glättung, rückwirkende Stoppbestätigung und Neustartprüfung verwenden Zukunftsinformation. Ein später Scoregewinn wäre keine Aussage über Echtzeitverhalten.

In [ ]:
def true_runs(mask):
    values = np.asarray(mask, dtype=bool)
    first = np.flatnonzero(values & ~np.r_[False, values[:-1]])
    last = np.flatnonzero(values & ~np.r_[values[1:], False])
    return list(zip(first, last))

def sustained(mask, seconds):
    length = max(1, round(seconds * FPS))
    return pd.Series(np.asarray(mask, bool)).rolling(length, min_periods=length).sum().eq(length).to_numpy()

tonal = (signals["tone_ratio"].ge(D_PARAMS["tone_ratio"])
         & signals["highband_ratio"].ge(D_PARAMS["highband_ratio"])
         & signals["tone_hz"].between(*D_PARAMS["tone_peak_hz"])
         & signals["audio_dbfs"].gt(D_PARAMS["min_audio_dbfs"])
         & signals["tone_hz"].diff().abs().le(D_PARAMS["tone_frequency_change_hz"]))
seed_rows = []
for source, mask in [("Tonaler Kandidat", tonal), ("Ball außerhalb", signals["outside"])]:
    for first, last in true_runs(mask):
        if last - first + 1 >= round(D_PARAMS["seed_min_seconds"] * FPS):
            seed_rows.append({"source": source, "first": first, "last": last})
seeds = pd.DataFrame(seed_rows, columns=["source", "first", "last"]).sort_values(["first", "source"]).reset_index(drop=True)
quiet = signals["active_fraction"].le(D_PARAMS["quiet_active_fraction"])
signals["quiet_confirmed"] = sustained(quiet, D_PARAMS["quiet_seconds"])
signals["stop_confirmation"] = signals["quiet_confirmed"] & (
    signals["speed_m_s"].le(D_PARAMS["slow_ball_m_s"]) | ~ball_seen)
ready = sustained(signals["speed_m_s"].le(D_PARAMS["ready_ball_m_s"]) & ~signals["outside"], D_PARAMS["ready_seconds"])
ready_recent = pd.Series(ready).rolling(round(D_PARAMS["ready_history_seconds"] * FPS), min_periods=1).max().astype(bool).to_numpy()
launch = sustained(signals["speed_m_s"].ge(D_PARAMS["launch_ball_m_s"]) & ~signals["outside"], D_PARAMS["launch_seconds"])
active = sustained(signals["active_fraction"].ge(D_PARAMS["restart_active_fraction"]), D_PARAMS["restart_active_seconds"])
inside = ball_seen & (ball_xy[:, 0] >= 0) & (ball_xy[:, 0] <= PITCH[0]) & (ball_xy[:, 1] >= 0) & (ball_xy[:, 1] <= PITCH[1])
restart_rows = []
lookahead = round(D_PARAMS["restart_lookahead_seconds"] * FPS)
for frame in np.flatnonzero(candidate_mask & ready_recent & inside):
    end = min(N_FRAMES, frame + lookahead + 1)
    launch_frames = np.flatnonzero(launch[frame:end]) + frame
    active_frames = np.flatnonzero(active[frame:end]) + frame
    if len(launch_frames) and len(active_frames):
        restart_rows.append({"restart_frame": frame, "confirmed_frame": int(max(launch_frames[0], active_frames[0]))})
restarts = pd.DataFrame(restart_rows, columns=["restart_frame", "confirmed_frame"])
display(seeds["source"].value_counts().rename("Hinweise im Gesamtvideo").to_frame())
print(f"{len(restarts)} mögliche Neustartkontakte vor Zustandsprüfung.")

In [ ]:
def infer_additional_dead(signals, seeds, restarts, halves):
    added_dead = np.zeros(len(signals), dtype=bool)
    records = []
    confirmations = np.flatnonzero(signals["stop_confirmation"].to_numpy())
    confirm_window = round(D_PARAMS["confirmation_seconds"] * FPS)
    maximum_dead = round(D_PARAMS["max_added_dead_seconds"] * FPS)
    seed_support = round(D_PARAMS["seed_min_seconds"] * FPS)
    quiet_support = round(D_PARAMS["quiet_seconds"] * FPS)
    for half, half_first, half_last in halves:
        blocked_until = half_first - 1
        for seed in seeds.loc[seeds["first"].between(half_first, half_last)].to_dict("records"):
            first = seed["first"]
            if first <= blocked_until:
                continue
            earliest_confirmation = first + max(seed_support, quiet_support) - 1
            eligible = confirmations[(confirmations >= earliest_confirmation)
                                     & (confirmations <= min(first + confirm_window, half_last))]
            if not len(eligible):
                continue
            confirmed = int(eligible[0])
            fallback = min(first + maximum_dead, half_last + 1)
            options = restarts.loc[restarts["restart_frame"].gt(confirmed)
                                   & restarts["restart_frame"].lt(fallback)
                                   & restarts["confirmed_frame"].le(half_last)]
            if len(options):
                restart = options.iloc[0]
                end, reason = int(restart["restart_frame"]), "Neustartregel"
                end_confirmation = int(restart["confirmed_frame"])
            else:
                end = fallback
                reason = "Halbzeitende" if end == half_last + 1 else "45-s-Fallback"
                end_confirmation = None
            added_dead[first:end] = True
            blocked_until = end - 1
            records.append({"half": half, "source": seed["source"], "start_frame": first,
                            "stop_confirmed_frame": confirmed, "end_exclusive": end,
                            "end_reason": reason, "restart_confirmed_frame": end_confirmation,
                            "duration_s": (end - first) / FPS,
                            "stop_confirmation_delay_s": (confirmed - first) / FPS})
    return added_dead, pd.DataFrame(records)

additional_dead, state_intervals = infer_additional_dead(signals, seeds, restarts, HALVES)
new_dead = (baseline_labels == "D") | additional_dead
baseline_with_dead = baseline_labels.copy()
baseline_with_dead[new_dead] = "D"
stoppage_labels, _ = classify_from_contacts(baseline_with_dead, candidate_side, HALVES)
assert np.array_equal(stoppage_labels == "D", new_dead), "Kontakt-Replay verändert die neue D-Maske."
assert np.all(stoppage_labels[baseline_labels == "D"] == "D"), "Alte D-Frames müssen erhalten bleiben."
predictions = {BASELINE_NAME: baseline_labels, CONTACT_NAME: contact_labels, METHOD_NAME: stoppage_labels}
for name, labels in predictions.items():
    pd.DataFrame({"frame": frame_ids, "label": labels}).to_csv(SUBMISSION_DIR / f"{name}.csv", index=False, lineterminator="\n")
state_intervals.to_csv(OUTPUT_DIR / "state_intervals.csv", index=False)
seeds.to_csv(OUTPUT_DIR / "stop_seeds.csv", index=False)
restarts.to_csv(OUTPUT_DIR / "restart_candidates.csv", index=False)
scenes.to_csv(OUTPUT_DIR / "review_scenes.csv")
prediction_hashes = {name: hashlib.sha256((SUBMISSION_DIR / f"{name}.csv").read_bytes()).hexdigest() for name in predictions}
display(state_intervals.groupby(["source", "end_reason"]).agg(Intervalle=("start_frame", "size"),
    Mittlere_Dauer_s=("duration_s", "mean")).round(2))
print(f"Zusätzliche D-Frames in den Halbzeiten: {np.sum(additional_dead & (baseline_labels != 'D')):,}.")
print("Alle drei vollständigen Vorhersagen gespeichert; noch kein GT-Score berechnet.")

## 8. Erst jetzt die öffentliche GT für den Vergleich laden

Wir prüfen weiterhin denselben öffentlichen Ausschnitt. Alle bekannten Beispiele haben die Problemwahl beeinflusst; daraus folgt kein unabhängiger Generalisierungsnachweis. Die Kontrollclips außerhalb des Ausschnitts haben keine veröffentlichte GT und erhalten hier keinen automatischen Wahrheitsstatus.

Die zusätzliche D-Maske kann nur D hinzufügen. H/A wird dennoch anschließend anders ausfallen können, weil Kontakte während D unterdrückt werden und die Teaminitialisierung nach D neu beginnt. Diese beiden Effekte werden getrennt sichtbar gemacht.

In [ ]:
with ZipFile(ARCHIVE) as official:
    gt = pd.read_csv(BytesIO(official.read(ARCHIVE_ROOT + "evaluation/gt/BAR-ATH_possession_gt_public.csv")))
    public_meta = json.loads(official.read(ARCHIVE_ROOT + "evaluation/gt/BAR-ATH_possession_gt_public_meta.json"))
assert np.array_equal(gt["frame"], frame_ids), "Öffentliche GT und Vorhersagen sind nicht framegleich."
graded = gt["label"].ne("X").to_numpy()
y_true = gt["label"].to_numpy()
validation_rows = []
with TemporaryDirectory(prefix="mtah06-official-") as temporary:
    with ZipFile(ARCHIVE) as official:
        official.extractall(temporary)
    official_root = Path(temporary) / ARCHIVE_ROOT
    for name in predictions:
        validated = subprocess.run([sys.executable, str(official_root / "evaluation/validate_submission.py"),
            str(SUBMISSION_DIR / f"{name}.csv")], capture_output=True, text=True, check=True)
        validation_rows.append({"Methode": name, "Validator": validated.stdout.splitlines()[0]})
    evaluation = subprocess.run([sys.executable, str(official_root / "evaluation/run_evaluation.py"),
        "--gt", "public", "--results", str(SUBMISSION_DIR), "--out", str(EVALUATION_DIR), "--no-detail"],
        capture_output=True, text=True, check=True)
(EVALUATION_DIR / "official_stdout.txt").write_text(evaluation.stdout, encoding="utf-8")
display(pd.DataFrame(validation_rows))
print(evaluation.stdout)

In [ ]:
results = pd.read_csv(EVALUATION_DIR / "scoreboard.csv").set_index("submission").loc[list(predictions)]
dead_rows = []
for name, labels in predictions.items():
    true_dead, predicted_dead = (y_true == "D") & graded, (labels == "D") & graded
    tp = int((true_dead & predicted_dead).sum())
    fp = int((~true_dead & predicted_dead & graded).sum())
    fn = int((true_dead & ~predicted_dead).sum())
    dead_rows.append({"submission": name, "D_precision": tp / (tp + fp) if tp + fp else 0,
                      "D_recall": tp / (tp + fn), "D_TP": tp, "D_FP_live": fp, "D_FN": fn})
dead_results = pd.DataFrame(dead_rows).set_index("submission")
comparison = results[["macro_f1", "spell_f1_mean", "poss", "f1_D"]].join(dead_results)
comparison.to_csv(OUTPUT_DIR / "public_comparison.csv")
display(comparison.round(4))
delta = comparison.loc[METHOD_NAME] - comparison.loc[CONTACT_NAME]
print(f"Gegenüber Notebook 04: macroF1 {delta['macro_f1']:+.4f}, spellF1 {delta['spell_f1_mean']:+.4f}, "
      f"D-Recall {delta['D_recall']:+.4f}, D-Precision {delta['D_precision']:+.4f}.")
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
plot_names = ["smooth50", "Kontakt 04", "Kontakt + D 06"]
plot_results = comparison.copy()
plot_results.index = plot_names
plot_results[["macro_f1", "spell_f1_mean"]].rename(columns={"macro_f1": "macroF1", "spell_f1_mean": "spellF1"}).plot.bar(ax=axes[0], rot=0)
plot_results[["D_precision", "D_recall"]].rename(columns={"D_precision": "D-Precision", "D_recall": "D-Recall"}).plot.bar(ax=axes[1], rot=0)
for ax, title in zip(axes, ["Offizieller öffentlicher Score", "Unterbrechungen: Präzision und Abdeckung"]):
    ax.set(title=title, xlabel="", ylabel="Score", ylim=(0, 1))
    ax.grid(axis="y", alpha=0.15)
plt.show()

### Welche neuen Fehler entstehen?

Die erste Tabelle zählt zusätzliche D-Frames in echtem D und in laufendem H/A. Die zweite zeigt die zusammenhängenden Live-Abschnitte, die die neue Regel zusätzlich zerschneidet. Ein hoher D-Recall alleine reicht nicht, wenn dadurch H/A-Spells verloren gehen. Veränderungen außerhalb der neuen D-Maske kommen ausschließlich vom zurückgesetzten Kontaktzustand.

In [ ]:
new_d = graded & (stoppage_labels == "D") & (contact_labels != "D")
changed_live = graded & (stoppage_labels != "D") & (contact_labels != "D") & (stoppage_labels != contact_labels)
display(pd.Series({"Zusätzlich korrekt D": int((new_d & (y_true == "D")).sum()),
                   "Zusätzlich fälschlich D in Live-H": int((new_d & (y_true == "H")).sum()),
                   "Zusätzlich fälschlich D in Live-A": int((new_d & (y_true == "A")).sum()),
                   "Geändertes H/A nach Zustandsreset": int(changed_live.sum()),
                   "Davon vorher falsch, jetzt richtig": int((changed_live & (contact_labels != y_true) & (stoppage_labels == y_true)).sum()),
                   "Davon vorher richtig, jetzt falsch": int((changed_live & (contact_labels == y_true) & (stoppage_labels != y_true)).sum())}, name="Öffentliche Frames").to_frame())
false_sections = pd.DataFrame([{"start_frame": first, "end_frame": last, "duration_s": (last - first + 1) / FPS}
                              for first, last in true_runs(new_d & (y_true != "D"))],
                             columns=["start_frame", "end_frame", "duration_s"])
display(false_sections.sort_values("duration_s", ascending=False).head(8).round(2))
false_sections.to_csv(OUTPUT_DIR / "new_false_dead_sections.csv", index=False)
dead_reference = pd.DataFrame([{"start_frame": first, "end_frame": last, "duration_s": (last - first + 1) / FPS}
                              for first, last in true_runs(graded & (y_true == "D"))])
display(dead_reference.assign(Recall_04=[np.mean(contact_labels[r.start_frame:r.end_frame + 1] == "D") for r in dead_reference.itertuples()],
                              Recall_06=[np.mean(stoppage_labels[r.start_frame:r.end_frame + 1] == "D") for r in dead_reference.itertuples()]).round(3))

### Start- und Endfehler getrennt sichtbar machen

Die folgende Diagnose ordnet jeder öffentlichen D-Phase das zusätzliche Regelintervall mit der größten Frame-Überlappung zu. Das ist eine lokale Erklärungshilfe, keine neue offizielle Matchingmetrik. Negative Startabweichung bedeutet: Die Regel stoppt zu früh. Negative Endabweichung bedeutet: Die Regel startet zu früh wieder. Ein fehlendes zusätzliches Intervall bedeutet nicht, dass die alte Baseline gar kein D erkannt hat; dafür stehen die Recall-Spalten oben.

Wir ändern nach dieser Tabelle keine Schwellen. Insbesondere darf ein frühes Tonsignal nicht rückwirkend als bewiesener Schiedsrichterpfiff bezeichnet werden.

In [ ]:
boundary_rows = []
for ref in dead_reference.itertuples():
    overlap = np.maximum(0, np.minimum(state_intervals["end_exclusive"], ref.end_frame + 1)
                         - np.maximum(state_intervals["start_frame"], ref.start_frame))
    if not overlap.gt(0).any():
        boundary_rows.append({"GT_start": ref.start_frame, "GT_end_exclusive": ref.end_frame + 1,
                              "source": "Kein zusätzliches überlappendes D"})
        continue
    inferred = state_intervals.loc[overlap.idxmax()]
    boundary_rows.append({"GT_start": ref.start_frame, "GT_end_exclusive": ref.end_frame + 1,
                          "source": inferred["source"], "rule_start": inferred["start_frame"],
                          "rule_end_exclusive": inferred["end_exclusive"],
                          "start_offset_s": (inferred["start_frame"] - ref.start_frame) / FPS,
                          "end_offset_s": (inferred["end_exclusive"] - ref.end_frame - 1) / FPS})
boundary_errors = pd.DataFrame(boundary_rows)
boundary_errors.to_csv(OUTPUT_DIR / "public_boundary_errors.csv", index=False)
display(boundary_errors.round(2))

## 9. Szenen mit Originalton und gleichen Signalspuren ansehen

Je Szene: GT (außerhalb öffentlich X), Kontaktregel 04, neue Regel 06; Ballgeschwindigkeit; aktive Spielerquote; tonaler Energieanteil. Orange markiert zusätzliche D-Intervalle, senkrechte Linien zeigen Stopp-Hinweise. Die grauen Standard-Tags werden darunter angezeigt, bleiben aber ohne Einfluss auf die Vorhersage.

Die eingebetteten Clips enthalten **Originalton**, auf Mono/AAC komprimiert. Im Player gegebenenfalls die Lautstärke erhöhen; die Quelle ist leise. Die Signalspur zeigt keine unabhängig erkannte Schiedsrichterentscheidung. Für eine neue manuelle Referenz in `control_review.csv`: erster D-Frame, erster wieder lebender Frame, Unsicherheit und weitere Unterbrechungen getrennt eintragen. Ein einziger Clip kann mehrere D-Intervalle enthalten.

In [ ]:
def show_scene(scene_id):
    scene = scenes.loc[scene_id]
    first, last = int(scene["first"]), int(scene["last"])
    idx = np.arange(first, last + 1)
    seconds = (idx - first) / FPS
    display(Markdown(f"### {scene_id} · {scene['reason']}\n\n"
                     f"Video **{first / FPS:.2f}–{(last + 1) / FPS:.2f} s**, Frames **{first}–{last}**. {scene['role']}."))
    window = signals.loc[idx]
    number = {label: i for i, label in enumerate(LABEL_ORDER)}
    raster = np.array([[number[x] for x in labels[idx]] for labels in [y_true, contact_labels, stoppage_labels]])
    fig, axes = plt.subplots(4, 1, figsize=(12, 7), sharex=True, constrained_layout=True,
                             gridspec_kw={"height_ratios": [1, 1, 1, 1]})
    axes[0].imshow(raster, aspect="auto", interpolation="nearest", cmap=ListedColormap(COLORS), vmin=0, vmax=3,
                   extent=(0, len(idx) / FPS, 2.5, -0.5))
    axes[0].set(yticks=range(3), yticklabels=["GT", "Kontakt 04", "Kontakt + D 06"], title="Labels und verfügbare Unterbrechungssignale")
    fig.legend(handles=[Patch(color=color, label=label) for color, label in zip(COLORS, LABEL_ORDER)],
               loc="outside upper right", ncol=4, fontsize=8)
    axes[1].plot(seconds, window["speed_m_s"], color="#1764ab", label="Ballgeschwindigkeit")
    axes[1].axhline(D_PARAMS["slow_ball_m_s"], color="0.5", linestyle="--", linewidth=0.8)
    axes[1].set(ylabel="Ball (m/s)")
    axes[2].plot(seconds, window["active_fraction"], color="#287c5c")
    axes[2].axhline(D_PARAMS["quiet_active_fraction"], color="0.5", linestyle="--", linewidth=0.8)
    axes[2].set(ylabel="Aktiver Anteil", ylim=(-0.02, 1.02))
    axes[3].plot(seconds, window["tone_ratio"], color="#744c9e", linewidth=0.8)
    axes[3].axhline(D_PARAMS["tone_ratio"], color="0.5", linestyle="--", linewidth=0.8)
    axes[3].set(ylabel="Ton-Energieanteil", xlabel=f"Sekunden ab Videosekunde {first / FPS:.2f}", ylim=(0, 1))
    local_seeds = seeds.loc[seeds["first"].between(first, last)]
    local_intervals = state_intervals.loc[(state_intervals["start_frame"] <= last) & (state_intervals["end_exclusive"] > first)]
    for ax in axes[1:]:
        for row in local_intervals.itertuples():
            ax.axvspan(max(0, (row.start_frame - first) / FPS), min(len(idx) / FPS, (row.end_exclusive - first) / FPS),
                       color="#e6aa43", alpha=0.18)
        for row in local_seeds.itertuples():
            ax.axvline((row.first - first) / FPS, color="#744c9e" if row.source == "Tonaler Kandidat" else "#c83e41", alpha=0.7, linewidth=0.8)
        ax.grid(alpha=0.15)
    plt.show()
    print("Neue D-Intervalle (Ende exklusiv; ein Fallback ist kein erkannter Neustart):")
    display(local_intervals[["source", "start_frame", "stop_confirmed_frame", "end_exclusive", "end_reason", "duration_s"]].round(2))
    print("Automatische Standard-Tags, nur zur Sichtung:")
    display(set_pieces.loc[(set_pieces["start_s"] <= (last + 1) / FPS) & (set_pieces["end_s"] > first / FPS),
                          ["id", "Team", "Type", "start_s", "end_s"]].round(2))
    capture = cv2.VideoCapture(str(VIDEO_PATH))
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.5), constrained_layout=True)
    for ax, frame in zip(axes, np.linspace(first, last, 3).astype(int)):
        capture.set(cv2.CAP_PROP_POS_FRAMES, frame)
        ok, image = capture.read()
        if not ok:
            capture.release()
            raise RuntimeError(f"Videoframe {frame} nicht lesbar.")
        ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        ax.set_title(f"{frame / FPS:.2f} s · GT {y_true[frame]} · neu {stoppage_labels[frame]}")
        ax.axis("off")
    capture.release()
    plt.show()
    with TemporaryDirectory(prefix="mtah06-video-") as temporary:
        clip = Path(temporary) / "scene.mp4"
        command = [FFMPEG, "-hide_banner", "-loglevel", "error", "-ss", str(first / FPS), "-i", str(VIDEO_PATH),
                   "-t", str((last - first + 1) / FPS), "-map", "0:v:0", "-map", "0:a:0?", "-vf", "scale=960:-2",
                   "-c:v", "libx264", "-preset", "veryfast", "-crf", "28", "-pix_fmt", "yuv420p",
                   "-c:a", "aac", "-ac", "1", "-b:a", "48k", "-movflags", "+faststart", str(clip)]
        subprocess.run(command, capture_output=True, text=True, check=True)
        display(Video(data=clip.read_bytes(), embed=True, mimetype="video/mp4", width=900,
                      html_attributes='controls preload="none"'))

In [ ]:
show_scene("dev_foul")

In [ ]:
show_scene("dev_goal_kick")

In [ ]:
show_scene("dev_pass")

In [ ]:
show_scene("dev_duel")

In [ ]:
show_scene("dev_box_live")

In [ ]:
show_scene("control_free_kick")

In [ ]:
show_scene("control_goal_kick")

In [ ]:
show_scene("control_throw_in")

### Sichtungsbefunde dieses Versuchs

- **Bekannte Unterbrechung nach Aktion:** Die zusätzliche D-Phase beginnt gegenüber der öffentlichen Referenz 2,12 s zu früh und endet 1,00 s zu früh. Ein früher tonaler Hinweis ist damit kein bewiesener exakter Pfiffzeitpunkt.
- **Bekannte Ballrückholung / Abstoßvorbereitung:** Der Beginn liegt 0,04 s vor der Referenz; die neue D-Phase endet 2,68 s zu spät. Die vorher vollständig verpasste Pause wird weitgehend erfasst, die Grenze zur Spielfortsetzung bleibt ungenau.
- **Live-Gegenproben:** Pass, enge Teamwechsel und die ergänzte Strafraumaktion sind Gegenproben auf unnötige Unterbrechungen. Bei der Strafraumszene wird die genaue Abschluss-/Abwehrart aus den Standbildern nicht als verifiziertes Kontaktlabel übernommen.
- **Neue Freistoßsichtung:** Der Clip zeigt eine Standardvorbereitung und die anschließende Aktion. Die Zusatzregel verändert den Anfang, später werden wieder D-Frames aus der bestehenden Baseline übernommen. Das gesamte D-Ergebnis kommt also nicht ausschließlich aus dem neuen Zustandsmodell.
- **Neue Abstoßsichtung:** Der ausgewählte Clip enthält auch eine Ecke; beide automatischen Tags überlappen. Das zusätzliche D-Intervall beginnt bereits vor dem Clip. Der Tagname alleine isoliert daher keine saubere Abstoßepisode.
- **Neue Einwurfsichtung:** Die zusätzliche D-Phase bleibt trotz schneller Ballbewegung und höherer Spieleraktivität über das Clipende hinaus bestehen und endet nur im 45-s-Fallback. Der Neustartfilter kann laufendes Spiel zu lange unterdrücken. Vor einer Bewertung der tatsächlichen Grenzfehler braucht dieser Clip eine manuelle Referenz.

Diese Befunde sind Diagnose und visuelle Sichtung, keine bereits fertig annotierte Kontrollmenge. Die exakten Zahlen oben stammen aus dem festgelegten Lauf; bei einer späteren Methodenänderung müssen auch diese Notizen neu geprüft werden.

## 10. Ergebnis festhalten, Grenzen sichtbar lassen

Die nachfolgenden Zahlen werden aus diesem Lauf berechnet. Die Regeln werden bei schlechtem Ergebnis nicht still nachjustiert. Insbesondere können Publikumstöne, Ballhöhe, unvollständige Spielertracks, schnelle Ausführungen und fehlende Ballmessungen Start und Ende verschieben.

Die drei neuen Kontrollclips sind **vorbereitete Sichtung**, keine bereits annotierte Validierung. Auch ein gefülltes Sichtungsblatt mit drei gezielt gewählten Standards ersetzt keine repräsentative Testmenge. Für die nächste Methodenentscheidung brauchen wir zusätzlich sicher laufende Abschlüsse/Abpraller und Unterbrechungen ohne auffälligen Ton oder Feldüberschreitung.

In [ ]:
public_intervals = state_intervals.loc[(state_intervals["start_frame"] <= int(frame_ids[graded][-1]))
                                       & (state_intervals["end_exclusive"] > int(frame_ids[graded][0]))]
display(Markdown(
    f"**Öffentlicher Vergleich:** D-Recall **{comparison.loc[CONTACT_NAME, 'D_recall']:.3f} → {comparison.loc[METHOD_NAME, 'D_recall']:.3f}**, "
    f"D-Precision **{comparison.loc[CONTACT_NAME, 'D_precision']:.3f} → {comparison.loc[METHOD_NAME, 'D_precision']:.3f}**. "
    f"spellF1 **{comparison.loc[CONTACT_NAME, 'spell_f1_mean']:.4f} → {comparison.loc[METHOD_NAME, 'spell_f1_mean']:.4f}**, "
    f"macroF1 **{comparison.loc[CONTACT_NAME, 'macro_f1']:.4f} → {comparison.loc[METHOD_NAME, 'macro_f1']:.4f}**.\n\n"
    f"**Fehlerbilanz:** {int((new_d & (y_true == 'D')).sum())} zusätzliche korrekt als D erkannte Frames, "
    f"{int((new_d & (y_true != 'D')).sum())} zusätzliche falsche D-Frames im Live-Spiel. "
    f"Im Gesamtspiel enden {int(state_intervals['end_reason'].eq('45-s-Fallback').sum())} neue D-Intervalle durch den Zeit-Fallback.\n\n"
    "**Interpretation:** Mehr D-Abdeckung muss gemeinsam mit Präzision, H/A-Fragmentierung und Grenzfehlern beurteilt werden. "
    "Die Heuristik bleibt eine Entwicklungsvariante und wird nicht automatisch zur Abgabe erklärt."
))
manifest = {"official_commit": config["official_commit"], "archive_sha256": config["archive_sha256"],
            "python": sys.version.split()[0], "packages": {name: version(name) for name in ["numpy", "pandas", "matplotlib", "opencv-python", "IPython"]},
            "input_sha256": INPUT_HASHES, "prediction_sha256": prediction_hashes, "n_frames": N_FRAMES, "fps": FPS,
            "contact_parameters": config, "dead_parameters": D_PARAMS, "parameter_search": False,
            "offline": True, "uses_future_information": True, "ground_truth": "public; development-contaminated",
            "tags_used_for_prediction": False, "audio_detector": "spectral tonal proxy, source unverified",
            "bootstrap": 0, "control_review": "control_review.csv; initially unannotated"}
(OUTPUT_DIR / "run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
unchanged = {name: hashlib.sha256((PROJECT_DIR / name).read_bytes()).hexdigest() == digest for name, digest in INPUT_HASHES.items()}
assert all(unchanged.values()), "Eine bisherige Eingabedatei wurde geändert."
display(pd.Series(unchanged, name="Bisherige Eingabe unverändert").to_frame())
print(f"Vorhersagen und Diagnose: {OUTPUT_DIR}")

**Nächster Schritt nach der gemeinsamen Sichtung:** Die neuen Kontrollclips mit Unsicherheitsgrenzen annotieren und die Abschluss-/Abwehrart der Live-Strafraumgegenprobe visuell bestätigen und weitere unabhängige Abschlüsse/Abpraller ergänzen. Erst anhand dieser Fälle entscheiden, ob Audio, Gruppenbewegung oder die Neustartregel geändert werden soll. Ein visueller Schiedsrichterdetektor oder ein trainierter Audioklassifikator ist hier noch nicht umgesetzt.

**Referenz:** Offizielle Baseline und Auswertung aus dem lokal gespeicherten [Challenge-Repository](https://github.com/metrica-sports/more-than-a-hack-2027), Commit `3de6b60aeabd2475d95879a651d38b72bc4fba00`; der RTS-/Kontaktversuch ist eine eigene partielle Umsetzung der Ideen aus Link & Hoernig (2017), [Individual ball possession in soccer](https://doi.org/10.1371/journal.pone.0179953). Metriken beider Arbeiten sind nicht direkt vergleichbar. Die eingebetteten Matchclips sind für lokale Analyse bestimmt; vor Veröffentlichung Notebook-Ausgaben entfernen und die Datennutzungsbedingungen beachten.